# Football Impact Rating – End‑to‑End Analysis

## Why Goals and Assists Are Not Enough

The Premier League’s official stats page is full of clean numbers: goals, assists, clean sheets. These are **terminal events** – the final touches in a long chain of decisions, movements, and actions that actually determined the outcome. They are the *consequence* of football, not the *cause*.

Consider two central midfielders from a hypothetical season:

- **Player A**: 8 goals, 6 assists in 38 games. The public narrative: excellent season.
- **Player B**: 1 goal, 3 assists in 38 games. The public narrative: quiet season.

Now look at their per‑90 data (the real stuff):
- Player A: 4.2 progressive passes, 1.1 tackles won, 18 pressures (30% success).
- Player B: 7.8 progressive passes, 2.1 tackles won, 26 pressures (38% success), 0.19 xAG/90 vs 0.11.

Player B was more valuable. His contributions never appeared in the goals+assists column.

This notebook walks through the **Football Impact Rating** system – a composite metric that tries to capture the full picture: defensive work, ball progression, pressing intensity, chance creation, and ball retention. We’ll build everything from a synthetic dataset calibrated against real FBref Premier League 2023‑24 data, then cluster players into archetypes, and finally visualise the results in a way that any scout would recognise.

> **Target audience**: People who wanna see me try my best to use my Football and ML skills to do something in one of my favorite domain.

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTS & SETUP
# ─────────────────────────────────────────────────────────────

import sys
import warnings
warnings.filterwarnings('ignore')          # Suppress deprecation chatter

# Add parent directory to path so we can import our own src modules
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# Use a dark background for plots – matches modern football analytics aesthetics
matplotlib.rcParams['figure.facecolor'] = '#0D1117'

# Our custom modules (explained in the guide)
from src.data_generator import FootballDataGenerator
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.impact_scorer import ImpactScorer
from src.clustering import PlayerArchetypeClusterer
from src.visualizer import FootballVisualizer

print('All modules imported successfully')

## Stage 1: The Dataset

We don’t have a real FBref export here, so we **generate a synthetic dataset** of 500 players. The generator is calibrated so that each position has realistic means and standard deviations for every raw statistic (e.g., strikers average ~0.38 xG/90, centre‑backs ~0.04). This is done by fitting normal distributions to real 2023‑24 Premier League data and then drawing samples.

**Key insight:** Position identity drives the distribution. A generated striker should look nothing like a generated centre‑back in per‑90 stats – otherwise the data would be useless for position‑specific analysis.

We also inject 15 *outlier* players who spike in one specific dimension (e.g., a Haaland‑like xG or a TAA‑like progressive passing). These players should naturally rise to the top, acting as a sanity check for our later scoring system.

In [ ]:
# Instantiate the generator (seed ensures reproducibility)
generator = FootballDataGenerator()
raw_df = generator.generate(n_players=500, seed=42)

print(f'Dataset shape: {raw_df.shape}')
print(f'Position counts: {raw_df["position"].value_counts().to_dict()}')
print('\nAge distribution by position:')
print(raw_df.groupby('position')['age'].describe()[['mean', 'min', 'max']].round(1))

In [ ]:
# Quick visual check: xG/90 distributions for each outfield position
# Strikers (ST) should have much higher xG than defenders (CB)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='#0D1117')
outfield_positions = ['ST', 'CM', 'FB', 'CB']
colors = ['#00FF87', '#FFB800', '#FF6B9D', '#00C8FF']

for ax, pos, color in zip(axes, outfield_positions, colors):
    ax.set_facecolor('#0D1117')
    data = raw_df[raw_df['position'] == pos]['xG'].dropna()
    ax.hist(data, bins=20, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'{pos}\nxG/90 (mean={data.mean():.3f})', color='white', fontsize=10)
    ax.tick_params(colors='grey')
    for spine in ax.spines.values():
        spine.set_color('grey')
        spine.set_alpha(0.3)

fig.suptitle('xG/90 Distribution by Position — Position Identity Check', 
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Strikers have ~4x the xG rate of CBs – the data is position‑authentic.')

## Stage 2: Preprocessing — The Minimum Minutes Decision

Raw per‑90 stats for players with very few minutes are dominated by **sample variance**. A player who scores in both of his two appearances has a 1.0 goals/90 rate – but that tells us nothing about his true ability.

We apply a **900‑minute threshold** (≈10 full games), which is the FBref community standard. Below that, the numbers are too noisy.

**Winsorisation (capping outliers)** – we do **not** remove outliers; we cap them at the Tukey fences (Q1‑1.5*IQR, Q3+1.5*IQR). Why? In football analytics, outliers ARE the signal. A 35‑xG season is not a data error – it’s the most important data point. If we remove or clip them too aggressively, the later Min‑Max scaling will squeeze all other players into a narrow band.

In [ ]:
preprocessor = DataPreprocessor()
processed_df = preprocessor.run(raw_df, min_minutes=900)

removed = len(raw_df) - len(processed_df)
print(f'Players removed (< 900 mins): {removed}')
print(f'Players retained: {len(processed_df)}')
print('\nRetained position breakdown:')
print(processed_df['position'].value_counts())

# Split into separate DataFrames per position – we'll process each independently
position_dfs = preprocessor.separate_by_position(processed_df)

## Stage 3: Feature Engineering — From Raw Stats to Football Dimensions

Raw per‑90 stats are noisy, correlated, and position‑dependent. We aggregate related actions into **composite metrics** that map onto real football concepts. Each composite is a weighted sum of carefully chosen raw stats.

| Metric | Name | What it captures | Example components (weights) |
|--------|------|------------------|------------------------------|
| **PPI** | Possession Progression Index | Advancing the ball | progressive carries (1.0), progressive passes (0.7), passes into final third (0.8), progressive passes received (0.4) |
| **DAQ** | Defensive Action Quality | Winning the ball effectively | tackles + interceptions (1.0), pressures success rate (0.5), blocks (0.3) |
| **CCC** | Chance Creation Contribution | Creating dangerous chances | key passes (1.0), xA (1.2), crosses into box (0.6), shot‑creating actions (0.8) |
| **BRS** | Ball Retention Score | Keeping possession under pressure | pass completion % (1.0), carries into pressure (0.7), miscontrols (-0.5), dispossessed (-0.8) |
| **PII** | Pressing Intensity Index | Pressing with purpose | pressures (1.0), successful pressures (1.5), tackles in final third (2.0) |
| **TGI** | Threat Generation Index | Net goal threat (created minus errors) | xG (1.0), xA (0.8), errors leading to shot (-1.5), own goals (-3.0) |

The weights are based on football intuition and calibrated to make each dimension as independent as possible (we check correlation later).

In [ ]:
engineer = FeatureEngineer()
featured_dfs = engineer.run(position_dfs)

# Inspect central midfielders – they should have a balanced profile
cm_features = featured_dfs['CM'][['PPI', 'DAQ', 'CCC', 'BRS', 'PII']]
print('CM Composite Feature Statistics:')
print(cm_features.describe().round(3))

In [ ]:
# Correlation check: we want our composites to capture different aspects of the game.
# Ideally correlations are low (<0.5). If two composites are highly correlated,
# they are measuring the same underlying trait – we should merge or rethink them.

fig, ax = plt.subplots(figsize=(8, 6), facecolor='#0D1117')
ax.set_facecolor('#0D1117')

corr = cm_features.corr()
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)

features = corr.columns.tolist()
ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, color='white', fontsize=10)
ax.set_yticklabels(features, color='white', fontsize=10)

for i in range(len(features)):
    for j in range(len(features)):
        val = corr.iloc[i, j]
        txt = 'black' if -0.5 < val < 0.5 else 'white'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=txt, fontsize=9)

plt.colorbar(im, ax=ax)
fig.suptitle('CM Composite Feature Correlation Matrix\n(Low correlations confirm independent dimensions)', 
             color='white', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## Stage 4: Impact Scoring

We now combine the composite metrics into a single **0‑100 Impact Score** using **position‑specific weights**. The weights encode a football philosophy: for a centre‑back, defending (DAQ) matters most; for a striker, goal threat (TGI) dominates.

**Critical design choice:** Normalisation is done **within each position group**, not globally. A CB’s DAQ of 75 means she is in the top 25% of CBs on defending – not the top 25% of all players (which would be meaningless because strikers barely tackle). Within‑position normalisation preserves variation.

We use **MinMaxScaler** (0–1) because we want a bounded, human‑readable score. The formula is simple:
\[ \text{score} = \sum_{m \in \text{metrics}} (\text{normalized}_m \times w_m) \]
and then multiplied by 100.

The weights are:
- **ST**: TGI 40%, CCC 20%, PPI 15%, DAQ 5%, BRS 10%, PII 10%
- **CM**: DAQ 30%, PPI 25%, BRS 20%, PII 15%, CCC 10%
- **CB**: DAQ 35%, BRS 25%, PPI 20%, PII 10%, CCC 5%, TGI 5%
- **FB**: PPI 30%, DAQ 25%, BRS 20%, PII 15%, CCC 10%
- **GK**: (separate composite based on save %, claiming %, etc.)

In [ ]:
scorer = ImpactScorer()
scored_dfs = scorer.score_all_positions(featured_dfs)

# Summary statistics – note that means are ~50 because of within‑position normalisation
print('Impact Score Distributions by Position:')
print(f'{"Position":<8} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 45)
for pos, df in scored_dfs.items():
    scores = df['impact_score']
    print(f'{pos:<8} {scores.mean():>8.1f} {scores.std():>8.1f} {scores.min():>8.1f} {scores.max():>8.1f}')

## Stage 5: Archetype Clustering

Raw scores are useful, but scouts think in terms of **archetypes**: “box‑to‑box engine”, “deep‑lying playmaker”, “ball‑playing libero”. We use **K‑Means clustering** on the normalised composite metrics to automatically discover these patterns.

**Why K‑Means?**
- Player feature space is roughly Gaussian per position → K‑Means (spherical clusters) works well.
- We want 4–6 archetypes per position – enough to be meaningful but not overwhelming.
- DBSCAN would mark unusual players as noise – analytically wrong; we want them to define a cluster of their own (e.g., “unique outlier”).
- After clustering, we manually assign descriptive labels by inspecting the centroid feature rankings.

**Why StandardScaler for clustering but MinMaxScaler for scoring?**  
Clustering needs true variance preserved. A feature that naturally varies a lot (e.g., progressive passes) should have more geometric influence than one with small variance (e.g., aerials won). StandardScaler (z‑score) retains these relative differences. For scoring, we need a bounded 0–100 output that humans understand immediately – hence MinMax.

In [ ]:
clusterer = PlayerArchetypeClusterer()
clustered_dfs = clusterer.run(scored_dfs)

# Show archetype distribution for two key positions
cm_archetypes = clustered_dfs['CM']['archetype_label'].value_counts()
print('CM Archetype Distribution:')
print(cm_archetypes.to_string())

print('\nCB Archetype Distribution:')
cb_archetypes = clustered_dfs['CB']['archetype_label'].value_counts()
print(cb_archetypes.to_string())

In [ ]:
# Mean composite scores per CM archetype – the statistical "fingerprint"
cm_df = clustered_dfs['CM']
norm_cols = [c for c in cm_df.columns if c.endswith('_normalized')]
clean_names = {c: c.replace('_normalized', '') for c in norm_cols}

archetype_profile = cm_df.groupby('archetype_label')[norm_cols].mean() * 100
archetype_profile = archetype_profile.rename(columns=clean_names)
print('CM Archetype Profiles (mean normalized scores, 0-100):')
print(archetype_profile.round(1).to_string())

## Stage 6: Visualisations

We follow the **StatsBomb / Opta** visual style: dark background, vibrant accents, and clear labeling. This aesthetic is instantly recognisable to anyone in football analytics.

- **Radar chart** – the universal scout report format. A single player’s normalised scores are plotted on six axes, forming a polygon. The more area, the more complete the player.
- **Scatter plots** – compare two dimensions (e.g., DAQ vs PPI) to reveal playing styles.
- **Archetype heatmap** – shows the average profile of each cluster.
- **Impact score distribution** – ranks players and annotates their strongest component.

In [ ]:
viz = FootballVisualizer()

# Radar for the top CM by impact score
top_cm = cm_df.nlargest(1, 'impact_score').iloc[0]['player_name']
print(f'Generating radar for top CM: {top_cm}')
fig = viz.radar_chart(top_cm, cm_df)
plt.show()

In [ ]:
# Scatter: DAQ vs PPI for all CMs
# Quadrants help classify:
# - Top‑right: complete midfielder (high both)
# - Top‑left: defensive specialist (high DAQ, low PPI)
# - Bottom‑right: pure progressor (high PPI, low DAQ)
# - Bottom‑left: rotation player (low both)
fig = viz.position_scatter(cm_df, 'CM', 'DAQ', 'PPI')
plt.show()

In [ ]:
# Archetype heatmap – shows the 'fingerprint' of each CM archetype
fig = viz.archetype_profile_heatmap(cm_df, 'CM')
plt.show()

In [ ]:
# Top 20 CMs annotated with their strongest composite
fig = viz.impact_score_distribution(cm_df, 'CM')
plt.show()

## Stage 7: Player Cards

The player card is the system’s primary output for individual assessment. It answers: *“What does this player do well, what are they weak at, and where do they rank among their positional peers?”*

Crucially, the card shows the **reasons** behind the score, not just the number. For example: “73/100 because elite BRS and PPI, but below‑average DAQ – a ball‑player who is a defensive liability.”

In [ ]:
# Combine all players into one DataFrame for cross‑lookups
all_players = pd.concat(list(clustered_dfs.values()), ignore_index=True)

# Top striker card
top_st = clustered_dfs['ST'].nlargest(1, 'impact_score').iloc[0]['player_name']
st_card = scorer.generate_player_card(top_st, all_players)
print('=== STRIKER PLAYER CARD ===')
for k, v in st_card.items():
    print(f'  {k}: {v}')

In [ ]:
# Top CB card – should look completely different
top_cb = clustered_dfs['CB'].nlargest(1, 'impact_score').iloc[0]['player_name']
cb_card = scorer.generate_player_card(top_cb, all_players)
print('=== CENTRE‑BACK PLAYER CARD ===')
for k, v in cb_card.items():
    print(f'  {k}: {v}')

## The Cross‑Position Comparison Problem

One of the system’s most important outputs is the **warning** it generates when you try to compare players from different positions. This is an analytical safeguard.

A striker scoring 75 on PPI and a CM scoring 75 on PPI have **not** performed the same progressive actions. The striker’s 75 is relative to other strikers (who average lower PPI because progressing the ball is not their primary function); the CM’s 75 is relative to other CMs (who are the most active progressors in the squad).

The underlying raw stats are very different. Comparing position‑normalised scores across positions is like comparing a winger’s tackles to a CDM’s – the role context makes the number meaningless.

In [ ]:
# Demonstrate the cross‑position warning
top_st_name = clustered_dfs['ST'].nlargest(1, 'impact_score').iloc[0]['player_name']
top_cm_name = clustered_dfs['CM'].nlargest(1, 'impact_score').iloc[0]['player_name']

comparison = scorer.compare_players([top_st_name, top_cm_name], all_players)
print(comparison.to_string())

## Summary: What This System Tells Us That Box Scores Don’t

1. **A striker who generates 0.4 xG but makes an error leading to a 0.25 xG counter‑attack** has a net TGI contribution of only +0.15 – important for teams that play a high‑risk style.
2. **A pressing midfielder who presses 30 times per 90 but wins the ball back only 15% of the time** contributes far less PII than one who presses 18 times at 35% – volume without success is noise.
3. **A centre‑back’s BRS score rewards those who can play under pressure** – the premium modern coaches pay for ball‑playing CBs is baked into the weights.
4. **Archetype clustering confirms patterns scouts know intuitively** – the “Deep‑Lying Playmaker” cluster shows high BRS + PPI but low PII, exactly the Busquets/Fabinho statistical fingerprint.

None of this appears in a goals‑and‑assists column.

---

**Next steps**: You can take this notebook and adapt it to real data (e.g., from FBref via `soccerdata` or `StatsBombR`). The pipeline is modular – just replace the generator with a CSV reader and ensure the column names match. Happy analysing!